# 사람 배치·조작 오프라인 학습 view 구성
## 1. 왜 만드는가
기존 Human 기록을 **어디에 놓았는가**와 **어떻게·언제 조작했는가**로 나눕니다. 아직 모델을 학습하지 않습니다.

- placement: 저장된 보드·조각 snapshot → Human occupied-cell 정답. 실제 행동 직전 상태라는 보장은 없습니다.
- control: 이미 관측한 prefix → 다음 행동/기록 간격. 첫 행동 전 대기시간은 알 수 없습니다.
- oracleCondition: 사람이 실제 선택한 목표를 조건으로 주는 **진단 전용** 값입니다. 배치 모델 입력이 아닙니다.
- eligibility: 배치와 조작의 사용 가능 여부를 따로 기록합니다. 한쪽이 실패해도 다른 쪽을 버리지 않습니다.

**모두 실행 기본값은 가상 Smoke뿐입니다.** 작은 실제 Train과 Full은 별도 설정/실행/결과 섹션입니다.
새 로그 수집·게임 규칙 복제·fit·Test payload 읽기·runtime 연결은 없습니다.

[명세](../../../ai-harness/ml-experiments/experiments/human-placement-action-offline-views.md) · [offline-use-v1](../../../docs/ai-ml/contracts/human-placement-action-link.md#9-기존-로그의-학습-사용-범위--offline-use-v1)


## 2. 손으로 따라가는 예제
기록이 `move_left@100 → rotate@100 → hard_drop@250`라면:

| 관측 입력 | 다음 행동 정답 | 간격 정답 |
|---|---|---:|
| BOS (구간 시작) | move_left | null: 첫 대기시간 미지원 |
| move_left | rotate | 0ms |
| move_left → rotate, 지난 간격 [0] | hard_drop | 150ms |

BOS는 실제 게임 행동이 아닙니다. 미래 hard_drop·전체 조작 수·마지막 여부는 입력에 넣지 않습니다.
Sequence gap 뒤에서는 prefix를 새로 시작하되, 누락 구간의 대기시간이나 첫 행동 label을 합성하지 않습니다.


In [ ]:
import copy
import hashlib
import json
import os
import tempfile
from collections import Counter
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_action_offline_views.ipynb"
LINK_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_action_link_readiness.ipynb"
LINK_DEFS_SHA = "4814ffefdc5aa929106ed5676afe3789288633bab57c7c8dc9bfa54c0d839ce3"
EXPERIMENT_ID = "human-placement-action-offline-views"

def definition_source(path):
    return [{"id": c["id"], "source": "".join(c["source"])}
            for c in json.loads(path.read_text(encoding="utf-8"))["cells"]
            if c["cell_type"] == "code" and "definitions" in c.get("metadata", {}).get("tags", [])]

def definition_sha(path):
    data = (json.dumps(definition_source(path), ensure_ascii=False, sort_keys=True,
                       allow_nan=False, indent=2) + "\n").encode("utf-8")
    return hashlib.sha256(data).hexdigest()

assert definition_sha(LINK_NOTEBOOK) == LINK_DEFS_SHA
link = {"__name__": "offline_view_dependencies"}
for cell in definition_source(LINK_NOTEBOOK):
    exec(compile(cell["source"], str(LINK_NOTEBOOK), "exec"), link)
for name in ("require", "finite", "integer", "json_bytes", "sha256_file",
             "normalize_event", "extract_pairs", "physical_key", "repo_input",
             "checked_json", "reject_constant", "finite_float", "validate_quality"):
    globals()[name] = link[name]
link["dataset"]["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID
ACTIONS = link["dataset"]["shared"]["ACTIONS"]
COMPONENTS = ("placement_snapshot_eligible", "control_sequence_eligible",
              "timing_pair_eligible", "oracle_goal_conditioned")


In [ ]:
def eligibility(usable, reasons):
    return {"eligible": bool(usable), "reasons": sorted(set(reasons))}

def input_ok(item, piece_index):
    return (isinstance(item, dict) and isinstance(item.get("action"), str)
            and item["action"] in ACTIONS and item["action"] != "hold"
            and item.get("pieceIndex") == piece_index
            and all(integer(item.get(k)) for k in ("pieceIndex", "sequence", "tick"))
            and finite(item.get("createdAt")) and item["createdAt"] >= 0)

def control_view(inputs, event, line, player_number):
    """검증된 인접 시도 pair만 연결하고 누락 뒤의 prefix를 다시 시작한다."""
    if not isinstance(inputs, list):
        return [], {"INPUT_SEQUENCE_INVALID": 1}
    safe = [item if input_ok(item, event["pieceIndex"]) else {} for item in inputs]
    pairs, rejected = extract_pairs(safe, {**event, "line": line}, player_number)
    by_index = {p["currentInputIndex"]: p for p in pairs}
    examples, prefix, intervals = [], [], []
    for index, item in enumerate(safe):
        if not item:
            prefix, intervals = [], []
            continue
        if index == 0:
            examples.append({"currentInputIndex": 0,
                             "input": {"prefixActions": ["<BOS>"], "observedIntervalsMs": []},
                             "target": {"action": item["action"], "intervalMs": None}})
        if index in by_index:
            pair = by_index[index]
            require(prefix, "PREFIX_MISSING")
            examples.append({"currentInputIndex": index,
                             "input": {"prefixActions": prefix.copy(),
                                       "observedIntervalsMs": intervals.copy()},
                             "target": {"action": item["action"],
                                        "intervalMs": pair["recordedIntervalMs"]}})
            intervals.append(pair["recordedIntervalMs"])
            prefix.append(item["action"])
        else:
            prefix, intervals = [item["action"]], []
    reasons = Counter(reason for row in rejected for reason in row["reasons"])
    reasons["INPUT_RECORD_INVALID"] += sum(not item for item in safe)
    return examples, {k: v for k, v in sorted(reasons.items()) if v}

def make_segment(raw, line, reference, source_sha):
    """원본을 변경하지 않고 입력·정답·진단 조건과 과제별 적격 여부를 분리한다."""
    identity, event, payload, trust = normalize_event(raw)
    require(reference["sourceLine"] == line and event["eventType"] == "placement_decision"
            and event["gameId"] == reference["gameId"]
            and event["pieceIndex"] == reference["pieceIndex"], "REFERENCE_MISMATCH")
    require(event["schemaVersion"] in (1, 2) and payload.get("source") == "human",
            "SOURCE_CONTRACT_INVALID")
    require(type(reference.get("playerNumber")) is int and reference["playerNumber"] > 0
            and reference.get("split") in ("train", "validation"), "PLAYER_OR_SPLIT_INVALID")
    board, active, selected = (payload.get(k) for k in ("boardBefore", "activePiece", "selectedPlacement"))
    placement_reasons, target, snapshot = [], None, None
    try:
        key = physical_key(selected, selected=True)
        target = {"physicalClassKey": key,
                  "selectedCells": sorted(({"row": c["row"], "col": c["col"]} for c in selected["cells"]),
                                          key=lambda c: (c["row"], c["col"]))}
    except ValueError as error:
        placement_reasons.append(str(error))
    try:
        require(isinstance(board, list) and len(board) == 20 and all(
            isinstance(row, list) and len(row) == 10 and all(
                c is None or isinstance(c, str) and bool(c.strip()) for c in row)
            for row in board), "BOARD_INVALID")
        physical_key(active)
        if target is not None:
            require(active["pieceType"] == selected["pieceType"], "PIECE_TYPE_MISMATCH")
        snapshot = {"boardBefore": copy.deepcopy(board),
                    "activePiece": {k: copy.deepcopy(active[k])
                                    for k in ("pieceType", "rotation", "row", "col")}}
        snapshot["activePiece"]["cells"] = sorted(
            ({"row": c["row"], "col": c["col"]} for c in active["cells"]),
            key=lambda c: (c["row"], c["col"]))
    except ValueError as error:
        placement_reasons.append(str(error))
    inputs = payload.get("inputSequence")
    hold = isinstance(inputs, list) and any(isinstance(i, dict) and i.get("action") == "hold" for i in inputs)
    examples, pair_rejections = ([], {}) if hold else control_view(inputs, event, line, reference["playerNumber"])
    control_reasons = ["hold_deferred"] if hold else ([] if examples else ["NO_VALID_CONTROL_EXAMPLE"])
    if hold:
        placement_reasons.append("hold_deferred")
    pair_count = sum(e["target"]["intervalMs"] is not None for e in examples)
    flags = {
        "placement_snapshot_eligible": eligibility(snapshot is not None and target is not None
                                                   and not placement_reasons, placement_reasons),
        "control_sequence_eligible": eligibility(examples, control_reasons),
        "timing_pair_eligible": eligibility(pair_count, control_reasons or
                                            ([] if pair_count else ["NO_VALID_ADJACENT_PAIR"])),
        "oracle_goal_conditioned": eligibility(target is not None and examples and not hold,
                                               control_reasons + ([] if target else ["GOAL_UNAVAILABLE"])),
    }
    return {"schemaVersion": 1,
            "segmentId": hashlib.sha256(json_bytes([source_sha, list(identity), line])).hexdigest(),
            "lineage": {"sourceSha256": source_sha, "sourceLine": line, "gameId": event["gameId"],
                        "playerNumber": reference["playerNumber"], "pieceIndex": event["pieceIndex"],
                        "split": reference["split"], "schemaVersion": event["schemaVersion"],
                        "trust": trust if trust in ("client_reported", "server_authoritative") else "unspecified",
                        **{k: reference.get(k, "unspecified") for k in ("mode", "difficulty", "outcome")}},
            "support": {"policy": "offline-use-v1", "playerMapping": "frozen_readiness_binding",
                        "observation": "unverified", "ruleset": "unspecified",
                        "historicalBuild": "unverified", "stage": "unspecified",
                        "geometry": "stored_cells_shape_only", "reachability": "not_checked",
                        "actionState": "sequence_only"},
            "placement": {"inputSnapshot": snapshot, "target": target,
                          "offlineOnly": True, "inferenceEligible": False},
            "control": {"examples": examples, "pairRejections": pair_rejections},
            "oracleCondition": {"humanGoal": copy.deepcopy(target), "diagnosticOnly": True,
                                "wholePolicyEvidence": False},
            "eligibility": flags}

def build_views(records, references, source_sha):
    """같은 identity의 충돌은 모두 거부하고 완전 중복은 한 번만 보존한다."""
    segments, rejected, identities = [], [], {}
    for line, raw in sorted(records.items()):
        try:
            identity = normalize_event(raw)[0]
            identities.setdefault(identity, []).append((line, hashlib.sha256(json_bytes(raw)).hexdigest()))
        except ValueError as error:
            rejected.append({"sourceLine": line, "reasons": [str(error)]})
    for identity, occurrences in sorted(identities.items()):
        if len({digest for _, digest in occurrences}) != 1:
            rejected.extend({"sourceLine": line, "reasons": ["IDENTITY_CONFLICT"]}
                            for line, _ in occurrences)
            continue
        line = occurrences[0][0]
        rejected.extend({"sourceLine": n, "reasons": ["EXACT_DUPLICATE_IGNORED"]}
                        for n, _ in occurrences[1:])
        try:
            require(line in references, "REFERENCE_MISSING")
            segment = make_segment(records[line], line, references[line], source_sha)
            if any(segment["eligibility"][k]["eligible"] for k in COMPONENTS[:2]):
                segments.append(segment)
            else:
                rejected.append({"sourceLine": line, "segment": segment,
                                 "reasons": sorted({r for f in segment["eligibility"].values()
                                                    for r in f["reasons"]})})
        except ValueError as error:
            rejected.append({"sourceLine": line, "reasons": [str(error)]})
    return {"segments": sorted(segments, key=lambda s: s["lineage"]["sourceLine"]),
            "rejected": sorted(rejected, key=lambda r: r["sourceLine"])}


In [ ]:
RAW_BYTES = 650669388
INPUT_PATHS = {
    "raw": ("HUMAN_OFFLINE_RAW", link["dataset"]["DEFAULT_SOURCE"], link["RAW_SHA"]),
    "manifest": ("HUMAN_OFFLINE_MANIFEST", "data/datasets/human-timing-v1-20261024-full/manifest.json",
                 link["MANIFEST_SHA"]),
    "index": ("HUMAN_OFFLINE_INDEX", "data/ml-experiments/human-action-data-readiness/full-report.json",
              link["READINESS_REPORT_SHA"]),
    "quality": ("HUMAN_OFFLINE_QUALITY", "data/ml-experiments/human-game-quality-audit/full-report.json",
                link["QUALITY_REPORT_SHA"]),
}

def select_references(readiness, quality_games, groups, scope):
    """Test는 예약 ID만 참조하고 raw에서 파싱할 행은 먼저 확정한다."""
    require(set(groups) == {"train", "validation", "test"}, "SPLIT_KEYS_INVALID")
    require(all(isinstance(groups[k], list) and len(groups[k]) == len(set(groups[k]))
                for k in groups), "SPLIT_DUPLICATE")
    require(not any(set(groups[a]) & set(groups[b]) for a, b in
                    (("train", "validation"), ("train", "test"), ("validation", "test"))), "GAME_SPLIT_LEAKAGE")
    require([len(groups[k]) for k in ("train", "validation", "test")] == [149, 31, 33],
            "SPLIT_COUNT_CHANGED")
    allowed = {g: s for s in ("train", "validation") for g in groups[s]}
    references = {}
    for ref in readiness["detail"]["segments"]:
        game = ref["gameId"]
        if game not in allowed:
            continue
        require(game in quality_games and quality_games[game]["status"] == "include_candidate",
                "GAME_NOT_APPROVED")
        players = {p["playerNumber"]: p for p in quality_games[game]["humanPlayers"]}
        if ref["playerNumber"] not in players:
            continue
        line = ref["sourceLine"]
        require(type(line) is int and line > 0 and line not in references, "REFERENCE_AMBIGUOUS")
        q = quality_games[game]
        references[line] = {**ref, "split": allowed[game], "mode": q["mode"],
                            "difficulty": q["difficulty"],
                            "outcome": players[ref["playerNumber"]].get("outcome", "unspecified")}
    if scope == "train_sample":
        require(all(n in references and references[n]["split"] == "train" for n in link["SAMPLE_LINES"]),
                "SAMPLE_REFERENCE_MISSING")
        references = {n: references[n] for n in link["SAMPLE_LINES"]}
    require(references and not any(r["gameId"] in groups["test"] for r in references.values()),
            "TEST_PARSE_FORBIDDEN")
    return references

def read_selected_raw(path, references):
    """참조에 없는 줄은 byte만 통과시킨다. Test payload의 JSON 파싱은 하지 않는다."""
    records, traversed, lines = {}, 0, 0
    with path.open("rb") as handle:
        for line, data in enumerate(handle, 1):
            traversed += len(data)
            lines = line
            if line in references:
                records[line] = json.loads(data, parse_constant=reject_constant, parse_float=finite_float)
    require(set(records) == set(references), "RAW_LINE_MISSING")
    return records, {"rawBytesTraversed": traversed, "rawLinesTraversed": lines,
                     "selectedRawRecordsParsed": len(records), "testPayloadsParsed": 0,
                     "byteCountExcludesChecksumPasses": True}

def construct_real(scope):
    """고정 입력·split·Human 결합을 검사한 뒤 선택된 raw만 메모리 view로 구성한다."""
    require(scope in ("train_sample", "full"), "SCOPE_INVALID")
    paths = {key: repo_input(os.environ.get(env, default))
             for key, (env, default, _) in INPUT_PATHS.items()}
    watched = {paths[k]: item[2] for k, item in INPUT_PATHS.items()}
    watched[SOURCE_NOTEBOOK] = sha256_file(SOURCE_NOTEBOOK)
    watched[link["READINESS_NOTEBOOK"]] = link["EXPECTED_READINESS_SHA"]
    require(all(sha256_file(p) == digest for p, digest in watched.items()), "INPUT_SHA_MISMATCH")
    require(paths["raw"].stat().st_size == RAW_BYTES, "RAW_BYTES_MISMATCH")
    code_sha = definition_sha(SOURCE_NOTEBOOK)
    timing_definitions = [{"id": c["id"], "source": "".join(c["source"])}
                          for c in json.loads(link["DATASET_NOTEBOOK"].read_text(encoding="utf-8"))["cells"]
                          if c["cell_type"] == "code" and c["id"] in link["DEFINITION_IDS"]]
    require(hashlib.sha256(json_bytes(timing_definitions)).hexdigest() == link["EXPECTED_DEFS_SHA"],
            "TIMING_DEFINITIONS_CHANGED")
    require(definition_sha(LINK_NOTEBOOK) == LINK_DEFS_SHA, "DEPENDENCY_DEFINITIONS_CHANGED")
    metadata = {k: checked_json(paths[k], INPUT_PATHS[k][2]) for k in ("manifest", "index", "quality")}
    require(metadata["manifest"]["source"]["sha256"] == link["RAW_SHA"]
            and metadata["index"]["source"]["sha256"] == link["RAW_SHA"]
            and metadata["index"]["sourceUnchanged"] is True, "SOURCE_LINEAGE_INVALID")
    quality = validate_quality(metadata["quality"], link["RAW_SHA"], RAW_BYTES)
    groups = metadata["manifest"]["split"]["gameIds"]
    references = select_references(metadata["index"], quality, groups, scope)
    records, io = read_selected_raw(paths["raw"], references)
    require(io["rawLinesTraversed"] == 122095 and io["rawBytesTraversed"] == RAW_BYTES,
            "SOURCE_LINE_OR_BYTE_COUNT_CHANGED")
    bundle = build_views(records, references, link["RAW_SHA"])
    require(all(sha256_file(p) == digest for p, digest in watched.items())
            and definition_sha(SOURCE_NOTEBOOK) == code_sha
            and definition_sha(LINK_NOTEBOOK) == LINK_DEFS_SHA, "INPUT_OR_CODE_CHANGED")
    provenance = {"inputs": {k: {"path": str(paths[k].relative_to(ROOT)), "sha256": INPUT_PATHS[k][2]}
                             for k in paths},
                  "sourceNotebookSha256": watched[SOURCE_NOTEBOOK],
                  "definitionSha256": code_sha, "linkDefinitionSha256": LINK_DEFS_SHA,
                  "timingDefinitionSha256": link["EXPECTED_DEFS_SHA"],
                  "readinessNotebookSha256": link["EXPECTED_READINESS_SHA"],
                  "qualityPolicy": "quality-policy-v1", "splitSeed": link["dataset"]["SPLIT_SEED"],
                  "sourceBytes": RAW_BYTES, "sourceExpectedLineCount": 122095,
                  "reservedTestGameIds": groups["test"],
                  "candidateGameCounts": {s: len(groups[s]) for s in ("train", "validation")},
                  "selectedReferenceCounts": dict(sorted(Counter(r["split"] for r in references.values()).items())),
                  "io": io, "sourceUnchanged": True}
    return bundle, provenance


In [ ]:
def summarize(bundle, scope, provenance):
    """분모·과제별 적격 수를 집계한다. 학습 성능 지표는 계산하지 않는다."""
    rows = bundle["segments"]
    counts = {"parsedRecords": len(rows) + len(bundle["rejected"]), "retainedSegments": len(rows),
              "rejectedRecords": len(bundle["rejected"]),
              "games": len({s["lineage"]["gameId"] for s in rows}),
              "firstActionLabels": sum(e["currentInputIndex"] == 0 for s in rows for e in s["control"]["examples"]),
              "timingPairs": sum(e["target"]["intervalMs"] is not None for s in rows for e in s["control"]["examples"]),
              "zeroMsPairs": sum(e["target"]["intervalMs"] == 0 for s in rows for e in s["control"]["examples"]),
              "distinctStoredPlacementKeys": len({s["placement"]["target"]["physicalClassKey"] for s in rows
                                                 if s["placement"]["target"] is not None})}
    per_split = {}
    for split in ("train", "validation"):
        subset = [s for s in rows if s["lineage"]["split"] == split]
        per_split[split] = {"segments": len(subset), "usedGames": len({s["lineage"]["gameId"] for s in subset}),
                            **{k: sum(s["eligibility"][k]["eligible"] for s in subset) for k in COMPONENTS}}
    all_segments = rows + [r["segment"] for r in bundle["rejected"] if "segment" in r]
    reasons = {k: dict(sorted(Counter(reason for s in all_segments
                                     for reason in s["eligibility"][k]["reasons"]).items())) for k in COMPONENTS}
    return {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID,
            "status": "offline_views_constructed_in_memory", "policy": "offline-use-v1",
            "runMode": "full" if scope == "full" else "smoke", "scope": scope,
            "fullCohortCovered": scope == "full", "policyDecisionEligible": False,
            "canonicalResultEligible": False, "productionIntegration": False,
            "modelFitCount": 0, "dataConstructionComplete": False, "datasetPublished": False,
            "inferenceEligible": False, "testPayloadRead": False,
            "counts": counts, "perSplit": per_split, "componentReasons": reasons,
            "structuralRejectionReasons": dict(sorted(Counter(
                reason for r in bundle["rejected"] if "segment" not in r for reason in r["reasons"]).items())),
            "provenance": provenance}

def fresh_dataset_path(base, name):
    """새 revision 이름 하나만 허용해 source alias·상위 경로·기존 파일을 차단한다."""
    require(isinstance(name, str) and bool(name) and name not in (".", "..")
            and Path(name).name == name and "/" not in name and "\\" not in name, "DATASET_NAME_INVALID")
    base = Path(base).absolute()
    require(base.is_dir() and not any(p.is_symlink() for p in (base, *base.parents)), "OUTPUT_PARENT_SYMLINK")
    target = base / name
    require(not target.exists() and not target.is_symlink(), "OUTPUT_ALREADY_EXISTS_OR_SYMLINK")
    return target

def publish_dataset(bundle, report, base, name):
    """독점 디렉터리에 기록하고 manifest를 원자적으로 마지막 게시한다. 미완성 폴더는 dataset이 아니다."""
    target = fresh_dataset_path(base, name)
    target.mkdir()
    files = {}
    # shortcut: a failed I/O leaves a manifest-less directory; retry with a new revision after inspection.
    for split in ("train", "validation", "rejected"):
        filename = split + ".jsonl"
        values = bundle["rejected"] if split == "rejected" else (
            s for s in bundle["segments"] if s["lineage"]["split"] == split)
        count = 0
        with (target / filename).open("xb") as handle:
            for row in values:
                handle.write((json.dumps(row, ensure_ascii=False, sort_keys=True,
                                         allow_nan=False, separators=(",", ":")) + "\n").encode("utf-8"))
                count += 1
            handle.flush()
            os.fsync(handle.fileno())
        files[filename] = {"rows": count, "sha256": sha256_file(target / filename)}
    manifest = {**report, "status": "offline_dataset_constructed",
                "dataConstructionComplete": True, "datasetPublished": True, "files": files,
                "publicationContract": "manifest_last_exclusive_commit"}
    link["publish_report"](manifest, target / "manifest.json")
    return manifest

def save_smoke_report(report, output):
    if output:
        destination = Path(output)
        if not destination.is_absolute():
            destination = ROOT / destination
        require(destination.parent.is_dir(), "REPORT_PARENT_MISSING")
        checked = link["checked_paths"](ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(destination), "smoke")[1]
        link["publish_report"](report, checked)

def show_views(bundle, report):
    print(report["scope"], "| modelFitCount=0 | inferenceEligible=false")
    display(pd.DataFrame([report["counts"]]))
    display(pd.DataFrame(report["perSplit"]).T)
    reason_rows = [{"component": k, "reason": reason, "segments": n}
                   for k, counts in report["componentReasons"].items() for reason, n in counts.items()]
    display(pd.DataFrame(reason_rows, columns=["component", "reason", "segments"]))
    retained_ids = {s["segmentId"] for s in bundle["segments"]}
    diagnosed = bundle["segments"] + [r["segment"] for r in bundle["rejected"] if "segment" in r]
    bias_frame = pd.DataFrame([{**{k: s["lineage"][k] for k in ("split", "mode", "difficulty", "outcome")},
                                "retained": s["segmentId"] in retained_ids} for s in diagnosed])
    if not bias_frame.empty:
        display(bias_frame.groupby(["split", "mode", "difficulty", "outcome", "retained"],
                                       dropna=False).size().rename("segmentCount").reset_index())
    print("Bias table excludes structural failures without a linked segment:",
          sum("segment" not in r for r in bundle["rejected"]))
    examples = [{"sourceLine": s["lineage"]["sourceLine"], "inputIndex": e["currentInputIndex"],
                 "observedPrefix": " → ".join(e["input"]["prefixActions"]),
                 "pastIntervalsMs": e["input"]["observedIntervalsMs"],
                 "nextActionLabel": e["target"]["action"], "intervalLabelMs": e["target"]["intervalMs"]}
                for s in bundle["segments"] for e in s["control"]["examples"][:3]][:12]
    example_frame = pd.DataFrame(examples)
    if not example_frame.empty:
        example_frame["intervalLabelMs"] = pd.array(example_frame["intervalLabelMs"], dtype="Float64")
    display(example_frame)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    pd.DataFrame(report["perSplit"]).T[list(COMPONENTS)].plot.bar(ax=axes[0], rot=0)
    axes[0].set(ylabel="eligible segment count", title=report["scope"] + " — per-task eligibility")
    actions = Counter(e["target"]["action"] for s in bundle["segments"] for e in s["control"]["examples"])
    if actions:
        pd.Series(dict(sorted(actions.items()))).plot.bar(ax=axes[1], rot=30)
    axes[1].set(ylabel="action target count (BOS + adjacent pairs)", title="recorded targets, not predictions")
    plt.tight_layout(); plt.show(); plt.close(fig)
    print("Snapshot association / attempted actions only; no replay, fit, or gameplay evidence.")


## 3. 가상 Smoke — 내부 검사와 작은 결과
실제 raw/Train/Validation/Test 파일을 열지 않습니다. 아래 assertions는 합성 자료에서
입력·정답 분리, BOS/0ms, gap, 부분 실패, Hold, 중복/충돌, Test 파싱 방지, immutable 출력을 확인합니다.


In [ ]:
def synthetic_bundle():
    records, references, _ = link["synthetic_case"]()
    references[1]["split"] = "train"
    return records, references

def self_check():
    records, refs = synthetic_bundle()
    original = json_bytes(records)
    normal = build_views(records, refs, "synthetic")
    segment = normal["segments"][0]
    examples = segment["control"]["examples"]
    assert len(examples) == 3
    assert [e["target"]["intervalMs"] for e in examples] == [None, 0, 150]
    assert examples[0]["input"]["prefixActions"] == ["<BOS>"]
    assert examples[-1]["input"] == {"prefixActions": ["move_left", "rotate"], "observedIntervalsMs": [0]}
    assert set(segment["placement"]["inputSnapshot"]) == {"boardBefore", "activePiece"}
    assert segment["placement"]["target"] == segment["oracleCondition"]["humanGoal"]
    assert "humanGoal" not in segment["placement"]["inputSnapshot"]
    assert segment["placement"]["inferenceEligible"] is False
    changed = copy.deepcopy(records)
    changed[1]["payload"]["selectedPlacement"]["rotation"] = 2
    assert build_views(changed, refs, "synthetic")["segments"][0]["placement"]["target"] == segment["placement"]["target"]
    gap = copy.deepcopy(records)
    gap[1]["payload"]["inputSequence"][1]["sequence"] = 7
    gap[1]["payload"]["inputSequence"][2]["sequence"] = 8
    gap_example = build_views(gap, refs, "synthetic")["segments"][0]["control"]["examples"][-1]
    assert gap_example["input"] == {"prefixActions": ["rotate"], "observedIntervalsMs": []}
    bad_board = copy.deepcopy(records)
    bad_board[1]["payload"]["boardBefore"] = None
    partial = build_views(bad_board, refs, "synthetic")["segments"][0]
    assert not partial["eligibility"]["placement_snapshot_eligible"]["eligible"]
    assert partial["eligibility"]["control_sequence_eligible"]["eligible"]
    bad_input = copy.deepcopy(records)
    bad_input[1]["payload"]["inputSequence"] = None
    partial = build_views(bad_input, refs, "synthetic")["segments"][0]
    assert partial["eligibility"]["placement_snapshot_eligible"]["eligible"]
    assert not partial["eligibility"]["control_sequence_eligible"]["eligible"]
    hold = copy.deepcopy(records)
    hold[1]["payload"]["inputSequence"][0]["action"] = "hold"
    isolated = build_views(hold, refs, "synthetic")
    assert not isolated["segments"] and "hold_deferred" in isolated["rejected"][0]["reasons"]
    duplicates = {1: records[1], 2: copy.deepcopy(records[1])}
    duplicate_refs = {**refs, 2: {**refs[1], "sourceLine": 2}}
    assert len(build_views(duplicates, duplicate_refs, "synthetic")["segments"]) == 1
    duplicates[2]["payload"]["boardBefore"] = None
    assert not build_views(duplicates, duplicate_refs, "synthetic")["segments"]
    assert json_bytes(build_views(dict(reversed(list(duplicates.items()))), duplicate_refs, "synthetic")) == json_bytes(
        build_views(duplicates, duplicate_refs, "synthetic"))
    assert json_bytes(records) == original
    second = copy.deepcopy(records[1])
    second["event_id"] = "second-event"
    second["game_id"] = "synthetic-validation"
    second_ref = {**refs[1], "sourceLine": 2, "gameId": "synthetic-validation", "split": "validation"}
    two_records, two_refs = {1: records[1], 2: second}, {**refs, 2: second_ref}
    two = build_views(two_records, two_refs, "synthetic")
    assert len(two["segments"]) == 2
    assert json_bytes(two) == json_bytes(build_views(dict(reversed(list(two_records.items()))), two_refs, "synthetic"))
    groups = {"train": ["synthetic-train"] + [f"train-{n}" for n in range(148)],
              "validation": ["synthetic-validation"] + [f"validation-{n}" for n in range(30)],
              "test": ["synthetic-test"] + [f"test-{n}" for n in range(32)]}
    quality = {g: {"status": "include_candidate", "humanPlayers": [{"playerNumber": 1}],
                   "mode": "pve", "difficulty": "normal"} for g in ("synthetic-train", "synthetic-validation")}
    index = {"detail": {"segments": [*two_refs.values(), {**refs[1], "sourceLine": 3, "gameId": "synthetic-test"}]}}
    assert set(select_references(index, quality, groups, "full")) == {1, 2}
    with tempfile.TemporaryDirectory() as directory:
        base = Path(directory).resolve()
        raw = base / "raw.jsonl"
        raw_line = json.dumps(records[1]).encode("utf-8") + b"\n"
        raw.write_bytes(raw_line + b"NOT_JSON_TEST_PAYLOAD\n" + raw_line)
        selected, io = read_selected_raw(raw, {**refs, 3: {**refs[1], "sourceLine": 3}})
        assert len(selected) == 2 and io["rawLinesTraversed"] == 3 and io["testPayloadsParsed"] == 0
        report = summarize(normal, "synthetic", {"sourceUnchanged": True})
        manifest = publish_dataset(normal, report, base, "revision1")
        assert manifest["files"]["train.jsonl"]["rows"] == 1 and (base / "revision1/manifest.json").is_file()
        for filename, info in manifest["files"].items():
            assert sha256_file(base / "revision1" / filename) == info["sha256"]
        from unittest.mock import patch
        with patch.object(os, "fsync", side_effect=OSError("synthetic_io_failure")):
            try:
                publish_dataset(normal, report, base, "incomplete")
            except OSError:
                pass
            else:
                raise AssertionError("IO_FAILURE_NOT_PROPAGATED")
        assert not (base / "incomplete/manifest.json").exists()
        for name in ("revision1", "../raw.jsonl", "raw.jsonl"):
            try:
                fresh_dataset_path(base, name)
            except ValueError:
                pass
            else:
                raise AssertionError("EXCLUSIVE_OUTPUT_GUARD_MISSING")
        (base / "alias").symlink_to(base / "revision1", target_is_directory=True)
        (base / "dangling").symlink_to(base / "missing")
        os.link(raw, base / "hardlink")
        for name in ("alias", "dangling", "hardlink"):
            try:
                fresh_dataset_path(base, name)
            except ValueError:
                pass
            else:
                raise AssertionError("OUTPUT_ALIAS_NOT_REJECTED")
        try:
            fresh_dataset_path(base / "alias", "new")
        except ValueError:
            pass
        else:
            raise AssertionError("SYMLINK_PARENT_NOT_REJECTED")
    return "PASS: separation / BOS / 0ms / gaps / partial / hold / identity / Test parse / publication"


In [ ]:
SMOKE_REPORT = os.environ.get("HUMAN_OFFLINE_SMOKE_REPORT") or None


In [ ]:
print(self_check())
smoke_records, smoke_refs = synthetic_bundle()
smoke_bundle = build_views(smoke_records, smoke_refs, "synthetic")
smoke_report = summarize(smoke_bundle, "synthetic", {
    "sourceNotebookSha256": sha256_file(SOURCE_NOTEBOOK),
    "definitionSha256": definition_sha(SOURCE_NOTEBOOK),
    "linkDefinitionSha256": LINK_DEFS_SHA, "sourceUnchanged": True,
    "io": {"selectedRawRecordsParsed": 0, "testPayloadsParsed": 0}})
save_smoke_report(smoke_report, SMOKE_REPORT)


In [ ]:
show_views(smoke_bundle, smoke_report)


## 4. 작은 실제 Train — 설정
먼저 위 Smoke 결과를 확인합니다. 아래 `RUN_TRAIN_SAMPLE=True`로 바꾸고 이 절의 실행/결과 셀을 실행합니다.
고정 Train sourceLine **118015/118021/118023**만 JSON 파싱합니다. 원본은 streaming 통과/해시 검사하므로 몇 초 걸릴 수 있습니다.
기존 timing pair lookup을 반복하지 않고 같은 parser·품질·인접 pair 정의를 사용합니다. **메모리 구성만** 하며 dataset 폴더는 만들지 않습니다.


In [ ]:
RUN_TRAIN_SAMPLE = True

In [ ]:
train_sample_bundle = train_sample_report = None
if RUN_TRAIN_SAMPLE:
    train_sample_bundle, train_sample_provenance = construct_real("train_sample")
    train_sample_report = summarize(train_sample_bundle, "train_sample", train_sample_provenance)
else:
    print("Small real Train skipped. Set RUN_TRAIN_SAMPLE=True in section 4.")


In [ ]:
if train_sample_report is not None:
    show_views(train_sample_bundle, train_sample_report)


## 5. Full — 별도 승인 후 실행
**기본 비활성**입니다. 작은 Train 결과를 검토하고 전체 생성 승인을 받은 뒤에만 실행합니다.
기존 Train 149/Validation 31게임의 승인 Human segment가 구성 후보입니다. 실제 적격 수는 생성 결과에서 확인합니다.
Test 33게임의 예약 ID는 manifest에만 보존하고 Test JSONL/raw payload는 파싱하지 않습니다.

1. `FULL_DATASET_NAME`에 data/datasets 아래의 **새 revision 폴더 이름 하나**를 적습니다. 미리 만들지 않습니다.
2. `RUN_FULL=True`로 바꾸고 이 절의 실행/결과 셀을 실행합니다.
3. train/validation/rejected JSONL과 마지막 게시되는 manifest를 확인합니다.
4. 기존 경로는 재사용할 수 없습니다. 실패해 manifest가 없는 폴더는 미완성입니다. 원인을 확인하고 새 revision으로 재시도합니다.

Manifest는 SHA·정의/정책·split·과제별 분모·제외 사유를 담습니다. 학습 모델/성능 report가 아닙니다.
전체 prefix를 명시적으로 보존하므로 긴 조작열에서는 메모리·출력 크기가 증가합니다. 관측되지 않은 정보를 합성하거나 긴 기록을 임의 절삭하지 않습니다.


In [ ]:
RUN_FULL = False
FULL_DATASET_NAME = "human-placement-action-offline-v1-20261102-run1"

In [ ]:
full_bundle = full_report = None
if RUN_FULL:
    fresh_dataset_path(ROOT / "data/datasets", FULL_DATASET_NAME)
    full_bundle, full_provenance = construct_real("full")
    full_report = summarize(full_bundle, "full", full_provenance)
    full_report = publish_dataset(full_bundle, full_report, ROOT / "data/datasets", FULL_DATASET_NAME)
else:
    print("Full skipped. Approval + a fresh FULL_DATASET_NAME + RUN_FULL=True are required.")


In [ ]:
if full_report is not None:
    show_views(full_bundle, full_report)
    display(pd.DataFrame(full_report["files"]).T)


## 6. 해석·한계·다음 gate
- 적격은 **제한된 offline-use-v1 과제의 형식 조건**입니다. action별 상태·게임 규칙·reachable 입력 검증 완료가 아닙니다.
- boardBefore는 고정 직전 저장 snapshot입니다. 행동마다 같았던 관찰 상태나 생각한 시간을 복원하지 않습니다.
- 첫 action은 학습 가능한 label이지만 첫 대기시간은 null입니다. Hold 포함 구간은 보류하고 원본과 게임은 보존합니다.
- Oracle 목표를 준 decoder 결과는 Human 배치 모델이나 전체 자율 policy 결과가 아닙니다.
- 다음은 작은 Train와 승인된 Full의 과제별 coverage 검토입니다. 이후 `prefix_only`와 `oracle_goal_prefix`의 학습 구조·예산을 **fit 전에 별도 명세**로 정합니다.
- Human 배치 선택 학습·예측 목표 실행·A/B 구조 비교는 별도 남은 단계입니다. 여기서는 fit·Test 평가·runtime·새 수집을 하지 않습니다.
